# Поведенческий тест MFT (CheckList)

MFT (minimum functionality, Ribeiro et al., ACL 2020): однозначные жалобы бытовым
языком с заранее известной группой → проверяем базовый навык модели. Считаем для
baseline и трансформера. **Runtime → GPU.**

In [ ]:
%cd /content
!rm -rf anamnesis && git clone -q https://github.com/vinograd131/anamnesis.git
%cd anamnesis
!pip install -q transformers accelerate peft
!pip uninstall -q -y torchao

## 1. Baseline (tf-idf + LogReg)

In [ ]:
from src.behavioral import baseline_predictor, run_mft
bl_proba, bl_classes = baseline_predictor()
run_mft(bl_proba, bl_classes, "baseline")

## 2. Трансформер (RuBioRoBERTa файнтюн)

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from src.transformer_ft import MODEL_ID, ID2LABEL, LABEL2ID, _predict_logits, _softmax
from src.mapping import GROUPS

tok = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(GROUPS), id2label=ID2LABEL, label2id=LABEL2ID
)
model = PeftModel.from_pretrained(base, "models/rubioroberta_ft")
device = "cuda" if torch.cuda.is_available() else "cpu"

def tf_proba(texts):
    return _softmax(_predict_logits(model, tok, texts, device, max_length=256))

run_mft(tf_proba, list(GROUPS), "rubioroberta_ft")

## 3. График

In [ ]:
from src.plot_behavioral import main as plot_behavioral
plot_behavioral()

## 4. Сохранить в git

In [ ]:
import getpass
token = getpass.getpass("GitHub token: ")
!git config user.email "karinkakarik@mail.ru"
!git config user.name "Git_Karina"
!git add reports/
!git commit -q -m "mft test"
!git pull -q --rebase https://github.com/vinograd131/anamnesis.git main
!git push -q https://{token}@github.com/vinograd131/anamnesis.git HEAD:main